# Oversight Arena — GRPO Training

Trains a small LLM overseer with TRL + Unsloth on the Oversight Arena
OpenEnv environment. Runs on a free T4 or local 12+ GB GPU.

Generates `results/loss_curve.png` and `results/reward_curve.png`.

In [ ]:
!git clone https://github.com/anikasoni/oversight-arena.git
%cd oversight-arena

In [ ]:
!pip install -q -U pip
!pip install -q -U 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q -U trl datasets accelerate bitsandbytes matplotlib
!pip install -q -e .

In [ ]:
# Smoke test: env works end-to-end
from oversight_arena import OversightArenaOpenEnv
env = OversightArenaOpenEnv(seed=0, difficulty=0.5)
obs = env.reset(seed=0)
print('workers:', obs['workers'])
print('diff (truncated):', obs['focused_patch_diff'][:400])
step = env.step({'action': 'flag_worker', 'worker_id': 'W2'})
print('reward after one flag:', step.reward)
print('grader:', env.grader())

In [ ]:
!python scripts/train_grpo.py \
    --model unsloth/Qwen2.5-1.5B-Instruct \
    --out checkpoints/grpo \
    --n-prompts 200 \
    --curriculum \
    --batch-size 2 \
    --grad-accum 4 \
    --num-generations 4

In [ ]:
from IPython.display import Image, display
display(Image('results/loss_curve.png'))
display(Image('results/reward_curve.png'))

In [ ]:
# Held-out eval: baseline vs trained
!python scripts/eval_llm.py --ckpt '' --label baseline --n-episodes 20
!python scripts/eval_llm.py --ckpt checkpoints/grpo --label grpo --n-episodes 20
!python scripts/plot_all.py